In [2]:
from typing import List, TypedDict, Literal
from pydantic import BaseModel, Field
import time

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
docs = (
    PyPDFLoader("../documents/Company_Policies.pdf").load() + 
    PyPDFLoader("../documents/Company_Profile.pdf").load() + 
    PyPDFLoader("../documents/Product_and_Pricing.pdf").load() 
)

In [4]:
chunks = RecursiveCharacterTextSplitter(
    chunk_size=600, 
    chunk_overlap=150, 
).split_documents(docs)

In [6]:
embeddings = OllamaEmbeddings(model='bge-m3')
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="../vector-database/selfrag3"
)

In [7]:
llm = ChatOllama(model="llama3.2", temperature=0)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})



In [8]:
# Graph state 
class State(TypedDict): 
    question: str
    need_retrieval: bool

    docs: List[Document]
    relevant_docs: List[Document]

    answer: str

In [9]:
class RetrieveDecision(BaseModel):
    should_retrieve: bool = Field(
        ...,
        description="True if external documents are needed to answer reliably, else False."
    )

decide_retrieval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You decide whether retrieval is needed.\n"
            "Return JSON that matches this schema:\n"
            "{{'should_retrieve': boolean}}\n\n"
            "Guidelines:\n"
            "- should_retrieve=True if answering requires specific facts, citations, or info likely not in the model.\n"
            "- should_retrieve=False for general explanations, definitions, or reasoning that doesn't need sources.\n"
            "- If unsure, choose True."
        ),
        ("human", "Question: {question}"),
    ]
)


# IMPORTANT: no `.content` for structured output
should_retrieve_llm = llm.with_structured_output(RetrieveDecision)

def decide_retrieval(state: "State"):
    decision: RetrieveDecision = should_retrieve_llm.invoke(
        decide_retrieval_prompt.format_messages(question=state["question"])
    )
    return {"need_retrieval": decision.should_retrieve}

In [10]:

direct_generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the question using only your general knowledge.\n"
            "Do NOT assume access to external documents.\n"
            "If you are unsure or the answer requires specific sources, say:\n"
            "'I don't know based on my general knowledge.'"
        ),
        ("human", "{question}"),
    ]
)


def generate_direct(state: State):
    out = llm.invoke(
        direct_generation_prompt.format_messages(
            question=state["question"]
        )
    )
    return {
        "answer": out.content
    }

In [11]:
def retrieve(state: State):
    return {"docs": retriever.invoke(state["question"])}

In [12]:
class RelevanceDecision(BaseModel): 
    is_relevant: bool = Field(
        ..., 
        description="True if the document helps answer the question, else False"
    )

In [15]:
is_relevant_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system", 
            "You are judging document relevance. \n"
            "Return JSON that matches this schema: \n"
            "{{'is_relevant': boolean}}\n\n"
            "A document is relevant if it contains information useful for answering the question."

        ), 
        (
            "human", 
            "Question:\n {question}\n\n Document: \n{document}"
        )
    ]
)

relevance_llm = llm.with_structured_output(RelevanceDecision)